In [ ]:
GRID3090

In [ ]:
镜像pytorch-2.6.0-py3.12-cuda12.4-u22.04:v3.0
GPU RTX 3090 * 1
CPU Intel(R) Xeon(R) Gold 6152 CPU * 10核
内存30GB
硬盘空出30~40GB 即可

In [ ]:
GRID-main文件夹新建data文件夹和flan_t5_xl文件夹，将对应的文件放进去，例如
data/amazon_data/sports
GRID-main/flan_t5_xl/model-00002-of-00002.safetensors

In [ ]:
先安装必要的库，需要去掉换行符，一行命令搞定

In [ ]:
pip install numpy==2.0.2 hydra-core==1.3.2 pytorch-lightning==2.5.2 transformers==4.47.0 pandas==2.3.0 accelerate==1.8.1 tfrecord==1.14.5 tensorboard==2.18.0 rootutils==1.0.7 lightning==2.5.0 pyarrow==20.0.0 tensorflow-cpu==2.18.0 hydra-colorlog==1.2.0 google-cloud-bigquery==3.29.0 huggingface-hub==0.33.2 tokenizers==0.21.0 scikit-learn

In [ ]:
 cd "/root/temp/GRID-main/"
命令一下：touch .project-root

In [ ]:
1. 加载本地的 Flan-T5 提取特征，将生成的张量保存下来（Inference）：

In [ ]:
执行命令可能因为单个显卡卡报错所以要改一下东西

In [ ]:
GRID-main/src/utils/inference_utils.py

In [ ]:
torch.distributed.barrier() 全部改成 
if torch.distributed.is_available() and torch.distributed.is_initialized():
    torch.distributed.barrier()

In [ ]:

本文的路径是/root/temp/GRID-main/你自己的路径如果不一样可以用记事本打开本ipynb
然后ctr+H, 将/root/temp替换成自己的路径，后面的一系列命令如果失败一般就是路径不符造成的
然后运行命令

In [ ]:
nohup python -u -m src.inference experiment=sem_embeds_inference_flat data_dir=data/amazon_data/sports embedding_model=/root/temp/GRID-main/flan_t5_xl > sem_embeds.log 2>&1 & 

In [ ]:
重新跑完后（大概等 6-7 分钟），你可以去新生成的时间戳目录下
（比如 logs/inference/runs/今天日期/新时间/pickle/）检查一下。如果里面出现了
一个叫 merged_predictions_tensor.pt 的文件，就说明大功告成！

In [ ]:
sem_embeds.log 大概倒数十几行会看到
wrote 53 rows to /root/temp/GRID-main/logs/inference/runs/2026-02-26/15-57-23
记住GRID-main/logs/inference/runs/2026-02-26/15-57-23这个路径后面embedding_path要用

In [ ]:
2读取刚才生成的张量（merged_predictions_tensor.pt）进行 RKmeans 聚类训练。（Train）：

In [ ]:
nohup python -u -m src.train experiment=rkmeans_train_flat \
  data_dir=data/amazon_data/sports \
  embedding_path=/root/temp/GRID-main/logs/inference/runs/2026-02-26/15-57-23/pickle/merged_predictions_tensor.pt \
  embedding_dim=2048 \
  num_hierarchies=3 \
  codebook_width=256 \
  > rkmeans_train.log 2>&1 &

In [ ]:
记住这个embedding_path，后面的embedding_path都是这个

In [ ]:
2分钟左右，当日志看到finished successfully.就表示跑完了
在日志里按ctr+f 找“checkpoints/checkpoint” (通常在倒数N行) 会看到类似
”GRID-main/logs/train/runs/2026-02-26/16-49-55/checkpoints/checkpoint_000_000030.ckpt' as top“
记住这个路径，下一步的ckpt_path 就是它

In [ ]:
3为商品分配离散化语义 ID (ID Assignment)230829
现在我们已经训练好了一个包含 3 层、每层 256 个簇的“语义字典”。接下来的任务是利用这个字典，
    把之前提取的商品连续特征（merged_predictions_tensor.pt）映射成确定的、离散的 
Semantic IDs（例如：商品A -> [12, 245, 87]）

In [ ]:
nohup python -u -m src.inference experiment=rkmeans_inference_flat \
  data_dir=data/amazon_data/sports \
  embedding_path=/root/temp/GRID-main/logs/inference/runs/2026-02-26/15-57-23/pickle/merged_predictions_tensor.pt \
  ckpt_path=/root/temp/GRID-main/logs/train/runs/2026-02-26/16-49-55/checkpoints/checkpoint_000_000030.ckpt \
  embedding_dim=2048 \
  num_hierarchies=3 \
  codebook_width=256 \
  > assign_ids.log 2>&1 &

In [ ]:
2分钟后从日志中可以看到：
Merged 18357 rows into merged_predictions_tensor.pt. as pytorch tensor

这说明你的数据集里一共有 18,357 个 sports 类目的商品，它们现在已经全部被成功映
射成了 3 层结构的离散化 Semantic IDs（语义 ID），并且没有任何越界报错。

至此，我们的**“前置三部曲”**已经全部顺利闯关成功：

sem_embeds_inference_flat（提取商品文本特征）✅

rkmeans_train_flat（训练残差 K-Means 聚类模型）✅

rkmeans_inference_flat（为所有商品分配最终的 Semantic IDs）✅

In [ ]:
4可以不跑，因为要跑的是改进后的，这里只是说一下原始模型正式训练生成式推荐模型 (Generative Recommender)

In [ ]:
nohup python -u -m src.train experiment=tiger_train_flat \
  data_dir=data/amazon_data/sports \
  semantic_id_path=/root/temp/GRID-main/logs/inference/runs/2026-02-25/17-05-22/pickle/merged_predictions_tensor.pt \
  num_hierarchies=4 \
  > tiger_train.log 2>&1 &

In [ ]:
模型架构加载成功：日志打印出了 TIGER 模型的结构（基于 T5 Encoder-Decoder 的架构），
总参数量为 13.1 M（1300万），这个量级在 3090 上跑起来会非常舒服。

损失值（Loss）正常下降：一开始的 Loss 大概在 23 左右，到了第 400 步时已经稳步下降到了 16.7 附近。
这说明模型正在努力学习 Semantic IDs 的序列规律！

多卡/单卡策略适配成功：系统自动识别到了你的单卡 3090，并启用了 bfloat16 混合精度训练，
这能大幅节省显存并加快训练速度。

In [ ]:
=====================================================================

In [ ]:
5魔改vq_vae

In [ ]:
src/modules/clustering/vq_vae_quantization.py

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning.pytorch as pl
from src.models.components.interfaces import OneKeyPerPredictionOutput

class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, commitment_cost=0.25):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost

        self.embedding = nn.Embedding(self.num_embeddings, self.embedding_dim)
        self.embedding.weight.data.uniform_(-1/self.num_embeddings, 1/self.num_embeddings)

    def forward(self, inputs):
        flat_inputs = inputs.view(-1, self.embedding_dim)
        
        distances = (torch.sum(flat_inputs**2, dim=1, keepdim=True) 
                    + torch.sum(self.embedding.weight**2, dim=1)
                    - 2 * torch.matmul(flat_inputs, self.embedding.weight.t()))

        encoding_indices = torch.argmin(distances, dim=1).unsqueeze(1)
        encodings = torch.zeros(encoding_indices.shape[0], self.num_embeddings, device=inputs.device)
        encodings.scatter_(1, encoding_indices, 1)
        
        quantized = torch.matmul(encodings, self.embedding.weight).view(inputs.shape)

        e_latent_loss = F.mse_loss(quantized.detach(), inputs)
        q_latent_loss = F.mse_loss(quantized, inputs.detach())
        vq_loss = q_latent_loss + self.commitment_cost * e_latent_loss

        quantized = inputs + (quantized - inputs).detach()

        return quantized, vq_loss, encoding_indices.view(inputs.shape[:-1])


class VQVAEQuantization(pl.LightningModule):
    # 核心改动 1：增加 **kwargs 吸收框架自动塞进来的多余参数
    def __init__(self, input_dim=2048, codebook_width=256, commitment_cost=0.25, lr=1e-3, **kwargs):
        super().__init__()
        self.save_hyperparameters()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.LayerNorm(input_dim),
            nn.ReLU()
        )
        self.quantizer = VectorQuantizer(codebook_width, input_dim, commitment_cost)
        
        self.decoder = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.LayerNorm(input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        quantized, vq_loss, semantic_ids = self.quantizer(z)
        reconstructed = self.decoder(quantized)
        return reconstructed, vq_loss, semantic_ids

    # 把特征提取逻辑单独抽出来，方便训练和预测共用
    def _extract_features(self, batch):
        x = None
        data_obj = batch[0] if isinstance(batch, (list, tuple)) else batch
        
        if hasattr(data_obj, 'transformed_features') and isinstance(data_obj.transformed_features, dict):
            x = data_obj.transformed_features.get('input_embedding')
        
        if x is None:
            for k in dir(data_obj):
                if not k.startswith('_'):
                    v = getattr(data_obj, k)
                    if isinstance(v, torch.Tensor) and v.is_floating_point() and self.hparams.input_dim in v.shape:
                        x = v
                        break
                        
        if x is None:
            raise ValueError("无法提取特征！")
            
        x = x.to(self.device)
        if x.dim() > 2:
            x = x.view(-1, x.shape[-1])
        return x.float()

    def training_step(self, batch, batch_idx):
        x = self._extract_features(batch)
        
        reconstructed, vq_loss, _ = self(x)
        recon_loss = F.mse_loss(reconstructed, x)
        loss = recon_loss + vq_loss
        
        self.log('train/loss', loss, prog_bar=True)
        self.log('train/recon_loss', recon_loss, prog_bar=True)
        self.log('train/vq_loss', vq_loss, prog_bar=True)
        
        return loss

    # 核心改动 2：专为 Inference 编写的预测步骤
    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        # 1. 提取特征并量化
        x = self._extract_features(batch)
        z = self.encoder(x)
        _, _, semantic_ids = self.quantizer(z)
        
        # 2. 终极自适应商品 ID 提取
        data_obj = batch[0] if isinstance(batch, (list, tuple)) else batch
        item_ids = None
        
        if hasattr(data_obj, 'item_ids') and data_obj.item_ids is not None:
            item_ids = data_obj.item_ids
        elif hasattr(data_obj, 'transformed_features') and isinstance(data_obj.transformed_features, dict):
            for possible_key in ['item_ids', 'id', 'item_id']:
                if possible_key in data_obj.transformed_features:
                    item_ids = data_obj.transformed_features[possible_key]
                    if item_ids is not None:
                        break
        
        if item_ids is None:
            for attr_name in dir(data_obj):
                if not attr_name.startswith('_'):
                    val = getattr(data_obj, attr_name)
                    if isinstance(val, torch.Tensor) and not val.is_floating_point() and val.shape[0] == x.shape[0]:
                        item_ids = val
                        break
        
        if item_ids is None:
            raise ValueError(f"彻底找不到 item_ids！Batch 结构: {dir(data_obj)}")
            
        # ==========================================
        # 核心改动：把 to(torch.int32) 改为 float()，迎合底层框架的 Float 合并机制！
        cluster_ids = semantic_ids.view(-1, 1).float()
        # ==========================================
        
        # 3. 使用框架的标准化输出结构
        return OneKeyPerPredictionOutput(
            keys=item_ids,
            predictions=cluster_ids,
            key_name="item_id",
            prediction_name="cluster_ids"
        )

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

In [ ]:
configs/experiment/vq_vae_train_flat.yaml

In [ ]:
# @package _global_
data_dir: ???
embedding_path: ???
embedding_dim: ???
num_hierarchies: ??? 
codebook_width: ??? 

task_name: train
id: ${now:%Y-%m-%d}/${now:%H-%M-%S}
tags:
- amazon-assign-ids-train-vqvae
train: true
test: false
ckpt_path: null
seed: 42

# ===== 数据加载部分 (保持与原版完全一致) =====
data_loading:
  features_config:
    features:
    - name: id
      num_placeholder_tokens: 0
      is_item_ids: true
      embeddings:
        _target_: torch.load
        _args_:
        - _target_: src.utils.file_utils.open_local_or_remote
          file_path: ${embedding_path}
          mode: rb
      type:
        _target_: torch.__dict__.get
        _args_:
        - int32
  dataset_config:
    dataset:
      _target_: src.data.loading.components.interfaces.ItemDatasetConfig
      item_id_field: id
      keep_item_id: true
      iterate_per_row: true
      data_iterator:
        _target_: src.data.loading.components.iterators.TFRecordIterator
      features_to_consider: ${extract_fields_from_list_of_dicts:${data_loading.features_config.features}, "name", False, "is_item_ids", "True"}
      embedding_map:
        id: ${data_loading.features_config.features[0].embeddings}
      num_placeholder_tokens_map: ${create_map_from_list_of_dicts:${data_loading.features_config.features}, "name", "num_placeholder_tokens"}
      preprocessing_functions:
      - _target_: src.data.loading.components.pre_processing.filter_features_to_consider
        _partial_: true
      - _target_: src.data.loading.components.pre_processing.convert_to_dense_numpy_array
        _partial_: true
      - _target_: src.data.loading.components.pre_processing.convert_fields_to_tensors
        _partial_: true
      - _target_: src.data.loading.components.pre_processing.map_sparse_id_to_embedding
        _partial_: true
        sparse_id_field: id
        embedding_field_to_add: embedding
      field_type_map: ${create_map_from_list_of_dicts:${data_loading.features_config.features}, "name", "type"}
  datamodule:
    _target_: src.data.loading.datamodules.sequence_datamodule.ItemDataModule
    train_dataloader_config:
      _target_: src.data.loading.components.interfaces.ItemDataloaderConfig
      dataset_class:
        _target_: src.data.loading.components.dataloading.UnboundedSequenceIterable
        _partial_: true
      data_folder: ${paths.data_dir}/items
      should_shuffle_rows: true
      batch_size_per_device:  512
      num_workers: 8
      assign_files_by_size: false
      timeout: 60
      drop_last: false
      pin_memory: true
      persistent_workers: true
      collate_fn:
        _target_: src.data.loading.components.collate_functions.collate_fn_items
        _partial_: true
        item_id_field: ${data_loading.dataset_config.dataset.item_id_field}
        feature_to_input_name:
          id: item_ids
          text: text_tokens
          text_mask: text_mask
          embedding: input_embedding # 注意这里重命名为了 input_embedding
      dataset_config: ${data_loading.dataset_config.dataset}
      limit_files: null
      assign_all_files_per_worker: true

# ===== 核心修改点 1：替换为 VQ-VAE 模型 =====
model:
  _target_: src.modules.clustering.vq_vae_quantization.VQVAEQuantization
  input_dim: ${embedding_dim}
  codebook_width: ${codebook_width}
  commitment_cost: 0.25
  lr: 0.001

callbacks:
  model_checkpoint:
    _target_: lightning.pytorch.callbacks.ModelCheckpoint
    dirpath: ${paths.output_dir}/checkpoints
    filename: checkpoint_{epoch:03d}_{step:06d}
    monitor: train/loss
    verbose: true
    save_last: true
    save_top_k: 1
    mode: min
    auto_insert_metric_name: false
    save_weights_only: false
    every_n_train_steps: null
    train_time_interval: null
    every_n_epochs: 1
  model_summary:
    _target_: lightning.pytorch.callbacks.RichModelSummary
    max_depth: -1

logger:
  csv:
    _target_: lightning.pytorch.loggers.csv_logs.CSVLogger
    save_dir: ${paths.output_dir}
    name: csv/
    prefix: ''

# ===== 核心修改点 2：将 max_steps 增加以适应神经网络训练 =====
trainer:
  _target_: lightning.pytorch.trainer.Trainer
  default_root_dir: ${paths.output_dir}
  min_steps: 1
  max_steps: 10000  # <--- VQVAE 需要更多步数进行反向传播
  max_epochs: 50
  accelerator: gpu
  devices: -1
  num_nodes: 1
  precision: bf16-mixed
  log_every_n_steps: 10
  val_check_interval: 100000000
  deterministic: false
  accumulate_grad_batches: 1
  profiler:
    _target_: lightning.pytorch.profilers.PassThroughProfiler
  strategy: ddp_find_unused_parameters_true
  sync_batchnorm: true
  num_sanity_val_steps: 0

paths:
  root_dir: .
  data_dir: ${data_dir}
  log_dir: ${paths.root_dir}/logs
  output_dir: ${hydra:runtime.output_dir}
  work_dir: ${hydra:runtime.cwd}
  profile_dir: ${hydra:run.dir}/profile_output
  metadata_dir: ${paths.output_dir}/metadata
extras:
  ignore_warnings: false
  enforce_tags: true
  print_config_warnings: true
  print_config: true
loss:
  loss_function: null
optim:
  optimizer: null
  scheduler: null
eval:
  evaluator: null

In [ ]:
embedding_path就是第1步的embedding_path

In [ ]:
nohup python -u -m src.train experiment=vq_vae_train_flat \
  data_dir=data/amazon_data/sports \
  embedding_path=/root/temp/GRID-main/logs/inference/runs/2026-02-26/15-57-23/pickle/merged_predictions_tensor.pt \
  embedding_dim=2048 \
  num_hierarchies=1 \
  codebook_width=256 \
  > vq_vae_train.log 2>&1 &

In [ ]:
10000步3090上1小时就能跑完）

In [ ]:
最终极优选 YAML 参数

In [ ]:
data_loading:
  datamodule:
    train_dataloader_config:
      batch_size_per_device: 512   # 黄金质量区间，保证梯度有足够的随机性探索
      num_workers: 8               # 保护你的 30G 内存不被撑爆
      pin_memory: true

trainer:
  precision: 32-true               # 放弃混合精度，使用纯 FP32 保证 L2 距离计算绝对精确！
  max_steps: 10000                 # 给模型充分的时间收敛
  val_check_interval: 100          # 如果有验证集，建议每100步看一次指标

In [ ]:
Loss 断崖式下降（极度收敛）：
 batch_size_per_device: 512 也可以改成2048但精度就不行了
在 2048.txt 中，训练到 130 多步时，总 Loss 还在 1.16 左右徘徊，下降非常缓慢。

在 512.txt 中，训练到 750 多步时，总 Loss 直接暴跌到了 0.103！

Reconstruction Loss（重建误差）几乎为 0：

日志显示 train/recon_loss=0.000167。这在自编码器（AutoEncoder）领域是一个满分级的表现。它意味着你的模型通过 256 个“离散的 ID（Codebook 向量）”，完美地还原了原本 2048 维的连续浮点数特征，几乎没有任何信息丢失！

训练节奏极其健康：

速度稳定在 3.18 it/s。由于 Batch Size 是 512，模型每秒能看 1600 多个商品，不仅运算效率高，而且梯度的“随机跳跃”帮助它轻松找到了最优解。

事实证明，512 的 Batch Size 绝对是当前数据量下 VQ-VAE 聚类的黄金甜点区！ 之前没有强行改 8000 是无比正确的决定。

In [ ]:
10000步3090上1小时就能跑完）。跑完后，
去你的输出目录下（logs/train/runs/对应的日期和时间/checkpoints/）
    找到那个刚刚出炉的、带着最高智慧结晶的checkpoint_000_010000.ckpt 权重文件。

比如logs/train/runs/2026-02-26/18-18-20/checkpoints/checkpoint_000_010000.ckpt 

In [ ]:
第二步：为商品分配最终的 Semantic IDs (推理阶段)
既然 VQ-VAE 已经学会了怎么把商品归类到 256 个坑位里，我们现在就要让它把全量（18000+个）商品的 ID 吐出来。

提前为你准备好下一步的命令：
等训练一结束，你就可以直接复制下面这行命令（注意替换你的 ckpt_path 和 embedding_path 为最新路径）：


In [ ]:
nohup python -u -m src.inference experiment=rkmeans_inference_flat \
  data_dir=data/amazon_data/sports \
  embedding_path=/root/temp/GRID-main/logs/inference/runs/2026-02-26/15-57-23/pickle/merged_predictions_tensor.pt \
  ckpt_path=/root/temp/GRID-main/logs/train/runs/2026-02-26/18-18-20/checkpoints/checkpoint_000_010000.ckpt \
  embedding_dim=2048 \
  num_hierarchies=1 \
  codebook_width=256 \
  model._target_=src.modules.clustering.vq_vae_quantization.VQVAEQuantization \
  > assign_vqvae_ids.log 2>&1 &

In [ ]:
然后2分钟左右，当你看到 Merged 18357 rows into merged_predictions_tensor.pt. as pytorch tensor 这行字
的时候，意味着我们亲手撸的这套 “端到端 VQ-VAE 离散语义量化系统” 已经完美无瑕地跑通了！

In [ ]:
这是日志的最后几行：

In [ ]:
Predicting DataLoader 0: |          | 144/? [00:24<00:00,  5.83it/s]
[[36m2026-02-26 21:11:55,944[0m][[34msrc.utils.inference_utils[0m][[32mINFO[0m] 
 - Global Rank: 0 wrote 18357 rows to /root/temp/GRID-main/logs/inference/runs/2026-02-26/21-11-21/pickle/predictions_0_20260226T131152954.pkl.[0m
[[36m2026-02-26 21:11:55,996[0m][[34msrc.utils.inference_utils[0m][[32mINFO[0m] - Rank 0 finished writing predictions.[0m
[[36m2026-02-26 21:11:55,996[0m][[34msrc.utils.inference_utils[0m][[32mINFO[0m] - Merging pickle files on main process.[0m
[[36m2026-02-26 21:12:03,869[0m][[34msrc.utils.inference_utils[0m][[32mINFO[0m] - Merged 18357 rows into merged_predictions.pkl.[0m
[[36m2026-02-26 21:12:05,537[0m][[34msrc.utils.inference_utils[0m][[32mINFO[0m] - Merged 18357 rows into merged_predictions_tensor.pt. as pytorch tensor[0m

Predicting DataLoader 0: |          | 144/? [00:37<00:00,  3.84it/s]

In [ ]:
记住倒数第N行rote 18357 rows to /root/tmp/GRID-main/logs/
runs/2026-02-26/21-11-21/pickle/predictions_0_20260226T131152954.pkl
这个runs/2026-02-26/21-11-21路径将进入下一步

In [ ]:
==========================================

In [ ]:
最终章 —— 训练 TIGER 生成式推荐模型

In [ ]:
nohup python -u -m src.train experiment=tiger_train_flat \
  data_dir=data/amazon_data/sports \
  semantic_id_path=/root/temp/GRID-main/logs/inference/runs/2026-02-26/21-11-21/pickle/merged_predictions_tensor.pt \
  num_hierarchies=2 \
  > tiger_train.log 2>&1 &

In [ ]:
你去 logs/train/runs/2026-02-26/21-19-05/checkpoints/ 目录下，
找到那个文件名里步数最高、保留下来的 最好的 .ckpt 文件
比如checkpoint_epoch=000_step=000600.ckpt

In [ ]:
nohup python -u -m src.inference experiment=tiger_inference_flat \
  data_dir=data/amazon_data/sports \
  semantic_id_path=/root/lanyun-tmp/GRID-main/logs/inference/runs/2026-02-26/21-11-21/pickle/merged_predictions_tensor.pt \
  ckpt_path=/root/lanyun-tmp/GRID-main/logs/train/runs/2026-02-26/21-19-05/checkpoints/000_step=000600.ckpt \
  num_hierarchies=2 \
  > tiger_inference.log 2>&1 &